In [1]:
import json

input_file = "финал.json"
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

for item in data:
    item['auctions_count'] = len(item.get('auctions', []))

output_file = f"финал2221.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

total_auctions = sum(item.get('auctions_count', 0) for item in data)
print(f"Saved {output_file}")
print(f"Objects {len(data)}, auctions {total_auctions}")

Saved финал2221.json
Objects 156, auctions 5170


In [2]:
import pandas as pd

inns_to_remove = [
    "5517010406", "5501182791", "7451438544", "7842032944", "7451327210",
    "562500016413", "562501539198", "2543004700", "2536015690", "2536252290",
    "2543004643", "2540065920", "2531008234", "2531008227", "2543009539", "2523004500"
]

df = pd.read_excel("zakupki.xlsx")
df['inn'] = df['inn'].astype(str).str.strip()
mask = df['inn'].isin(inns_to_remove)
rows_to_delete = mask.sum()
df_cleaned = df[~mask]
df_cleaned.to_excel("zakupki_cleaned.xlsx", index=False, engine="openpyxl")
print(f"Removed {rows_to_delete} rows, {len(df_cleaned)} left")

Removed 75 rows, 52774 left


In [3]:
import pandas as pd
import json

with open("финал222.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

bad_auctions = set()
for item in data:
    bad_auctions.update(item.get("auctions", []))
print(f"Bad auctions {len(bad_auctions)}")

df = pd.read_excel("zakupki_cleaned.xlsx")
df['notif_num'] = df['notif_num'].astype(str).str.strip()
df = df[df['notif_num'].notna() & (df['notif_num'] != '') & (df['notif_num'] != 'nan')]
df['collusion'] = df['notif_num'].isin(bad_auctions)
df.to_excel("zakupki_with_collusion.xlsx", index=False, engine="openpyxl")
collusion_true = df['collusion'].sum()
print(f"Collusion {collusion_true}, no collusion {len(df) - collusion_true}")

Bad auctions 5203
Collusion 3228, no collusion 46665


In [4]:
import pandas as pd

df1 = pd.read_excel("part12345678.xlsx")
df2 = pd.read_excel("zakupki_with_collusion.xlsx")
df_merged = pd.merge(df1, df2, on="regnum", how='inner')
df_merged.to_excel("polniy_pochti.xlsx", index=False, engine="openpyxl")
print(f"Merged {len(df_merged)} rows")

Merged 18537 rows


In [5]:
import pandas as pd
from pathlib import Path

INPUT_PATH = 'polniy_pochti.xlsx'
OUTPUT_PATH = 'cleaned_dataset.csv'

ID_COLUMNS = [
    'regnum', 'regnum_searched', 'notif_num', 'number',
    'notification_purchase_number', 'purchase_code', 'customer_spz_code',
    'inn', 'inn_searched_x', 'inn_searched_y', 'customer_inn_x', 'customer_inn_y',
    'customer_kpp_x', 'customer_kpp_y', 'kpp', 'ogrn', 'customer_okpo',
    'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
    'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp',
    'suppliers_4_inn', 'suppliers_4_kpp',
]

def read_excel_safe(path):
    header = pd.read_excel(path, nrows=0)
    dtype_map = {c: str for c in ID_COLUMNS if c in header.columns}
    df = pd.read_excel(path, dtype=dtype_map)
    for c in dtype_map:
        df[c] = (df[c].astype(str)
                 .str.replace(r'\.0$', '', regex=True)
                 .replace({'nan': None, 'None': None, '': None}))
    return df

def clean_dataset(df):
    original_n_cols = df.shape[1]
    print(f"Start: {df.shape[0]} rows, {df.shape[1]} columns")
    df = df.copy()

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    df = df.drop(columns=empty_cols)

    if 'is_unfair' in df.columns and 'suppliers_0_is_unfair' in df.columns:
        df['is_unfair'] = df['is_unfair'].combine_first(df['suppliers_0_is_unfair'])
        df = df.drop(columns=['suppliers_0_is_unfair'])

    if 'amount_rur' in df.columns and 'price_rur' in df.columns:
        df = df.drop(columns=['price_rur'])

    duplicates_to_drop = [
        'fz_y', 'fz_searched', 'customer_fz', 'suppliers_0_fz',
        'regnum_searched', 'currency_y', 'sign_date_y', 'exec_start_date',
        'subject', 'industry_y', 'subindustry_y', 'region_code_x',
        'fed_district_code_y', 'customer_inn_y', 'customer_kpp_y', 'customer_name',
        'amount', 'inn_searched_x', 'inn_searched_y',
    ]
    existing_dups = [c for c in duplicates_to_drop if c in df.columns]
    df = df.drop(columns=existing_dups)

    rename_map = {
        'fz_x': 'fz',
        'sign_date_x': 'sign_date',
        'currency_x': 'currency',
        'region_name_x': 'contract_region_name',
        'region_code': 'contract_region_code',
        'fed_district_code_x': 'fed_district_code',
        'industry_x': 'industry',
        'subindustry_x': 'subindustry',
        'customer_inn_x': 'customer_inn',
        'customer_kpp_x': 'customer_kpp',
        'region_code_y': 'supplier_region_code',
        'region_name_y': 'supplier_region_name',
        'price': 'price_original_currency',
        'amount_rur': 'price_rur',
    }
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)

    print(f"Result: {df.shape[0]} rows, {df.shape[1]} columns (dropped {original_n_cols - df.shape[1]})")
    return df

df = read_excel_safe(INPUT_PATH)
cleaned = clean_dataset(df)
cleaned.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}")

Start: 18537 rows, 148 columns
Result: 18537 rows, 96 columns (dropped 52)
Saved cleaned_dataset.csv


In [6]:
import numpy as np
import pandas as pd
import ast

INPUT_PATH = 'cleaned_dataset.csv'
OUTPUT_PATH = 'features.csv'

ID_COLUMNS = [
    'regnum', 'notif_num', 'number', 'notification_purchase_number',
    'purchase_code', 'customer_spz_code', 'inn', 'customer_inn',
    'customer_kpp', 'kpp', 'ogrn', 'customer_okpo',
    'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
    'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp',
]

def prepare(df):
    df = df.copy()
    print(f"Start: {df.shape[0]} rows, {df.shape[1]} columns")

    date_cols = ['sign_date', 'publish_date', 'execution_start_date',
                 'execution_end_date', 'placement_date']
    for c in date_cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')

    leak_cols = [c for c in [
        'contracts44_count', 'contracts44_sum', 'contracts223_count', 'contracts223_sum',
        'contracts_count', 'contracts_sum', 'customer_totals_contracts44_count',
        'customer_totals_contracts44_sum', 'customer_totals_contracts223_count',
        'customer_totals_contracts223_sum', 'customer_totals_contracts_count',
        'customer_totals_contracts_sum'
    ] if c in df.columns]
    df = df.drop(columns=leak_cols)

    const_cols = []
    for c in df.columns:
        if c == 'collusion':
            continue
        if df[c].dropna().nunique() <= 1:
            const_cols.append(c)
    df = df.drop(columns=const_cols)

    if 'notification' in df.columns and 'notif_num' in df.columns:
        df = df.drop(columns=['notification'])

    dup_keys = [c for c in ['regnum', 'sign_date', 'price_rur', 'inn'] if c in df.columns]
    df = df.drop_duplicates(subset=dup_keys, keep='first').reset_index(drop=True)

    sort_keys = ['sign_date']
    if 'eis_url' in df.columns:
        sort_keys.append('eis_url')
    elif 'regnum' in df.columns:
        sort_keys.append('regnum')
    df = df.sort_values(sort_keys, kind='stable').reset_index(drop=True)
    return df

def add_contract_features(df):
    df = df.copy()
    print("Contract-level features")

    df['log_price'] = np.log1p(df['price_rur'])
    df['is_foreign_currency'] = (df['currency'] != 'RUB').astype('Int64')

    def round_factor(x):
        if pd.isna(x) or x == 0:
            return 0
        for power in [6, 5, 4, 3, 2, 1]:
            if x % (10 ** power) == 0:
                return power
        return 0
    df['price_round_factor'] = df['price_rur'].apply(round_factor)

    df['has_nmck'] = df['notification_max_price'].notna().astype('Int64')
    df['price_drop_abs'] = df['notification_max_price'] - df['price_rur']
    df['price_drop_pct'] = np.where(
        df['notification_max_price'] > 0,
        df['price_drop_abs'] / df['notification_max_price'],
        np.nan,
    )

    sd = df['sign_date']
    df['sign_year'] = sd.dt.year
    df['sign_month'] = sd.dt.month
    df['sign_quarter'] = sd.dt.quarter
    df['sign_dayofweek'] = sd.dt.dayofweek
    df['is_end_of_year'] = sd.dt.month.isin([11, 12]).astype('Int64')
    df['is_end_of_quarter'] = sd.dt.month.isin([3, 6, 9, 12]).astype('Int64')

    df['days_sign_to_publish'] = (df['publish_date'] - df['sign_date']).dt.days
    df['days_sign_to_placement'] = (df['placement_date'] - df['sign_date']).dt.days
    df['days_sign_to_exec_start'] = (df['execution_start_date'] - df['sign_date']).dt.days
    df['contract_duration_days'] = (df['execution_end_date'] - df['execution_start_date']).dt.days

    df['is_same_region_customer_supplier'] = (
        df['customer_region_code'].astype(str) == df['supplier_region_code'].astype(str)
    ).astype('Int64')
    df['is_contract_in_customer_region'] = (
        df['contract_region_code'].astype(str) == df['customer_region_code'].astype(str)
    ).astype('Int64')
    df['is_contract_in_supplier_region'] = (
        df['contract_region_code'].astype(str) == df['supplier_region_code'].astype(str)
    ).astype('Int64')

    supplier_inn_cols = [f'suppliers_{i}_inn' for i in range(5) if f'suppliers_{i}_inn' in df.columns]
    df['n_suppliers'] = df[supplier_inn_cols].notna().sum(axis=1)
    df['is_single_supplier'] = df['single_supplier_reason_code'].notna().astype('Int64')
    df['subject_length'] = df['contract_subject'].fillna('').str.len()

    def parse_products(s):
        if pd.isna(s):
            return []
        try:
            return ast.literal_eval(s) if isinstance(s, str) else s
        except:
            return []
    products = df['products'].apply(parse_products)
    df['n_products'] = products.apply(len)
    df['total_quantity'] = products.apply(lambda lst: sum((p.get('quantity') or 0) for p in lst if isinstance(p, dict)))

    def first_okpd2_section(s):
        if pd.isna(s):
            return np.nan
        try:
            codes = ast.literal_eval(s) if isinstance(s, str) else s
            if codes and isinstance(codes[0], str):
                return codes[0][:2]
        except:
            pass
        return np.nan
    df['okpd2_section'] = df['product_codes'].apply(first_okpd2_section)
    return df

def add_history_features(df):
    df = df.copy()
    print("History features")

    g_sup = df.groupby('inn', sort=False)
    df['supplier_n_contracts_before'] = g_sup.cumcount()
    df['supplier_avg_price_before'] = g_sup['price_rur'].apply(lambda s: s.shift().expanding().mean()).droplevel(0)
    df['supplier_median_price_before'] = g_sup['price_rur'].apply(lambda s: s.shift().expanding().median()).droplevel(0)
    df['supplier_std_price_before'] = g_sup['price_rur'].apply(lambda s: s.shift().expanding().std()).droplevel(0)
    df['supplier_max_price_before'] = g_sup['price_rur'].apply(lambda s: s.shift().expanding().max()).droplevel(0)

    def n_unique_before(s):
        seen = set()
        out = []
        for v in s:
            out.append(len(seen))
            seen.add(v)
        return pd.Series(out, index=s.index)
    df['supplier_n_unique_customers_before'] = g_sup['customer_inn'].apply(n_unique_before).droplevel(0)
    df['supplier_days_since_last_contract'] = g_sup['sign_date'].apply(lambda s: (s - s.shift()).dt.days).droplevel(0)
    df['supplier_avg_price_drop_before'] = g_sup['price_drop_pct'].apply(lambda s: s.shift().expanding().mean()).droplevel(0)

    g_cust = df.groupby('customer_inn', sort=False)
    df['customer_n_contracts_before'] = g_cust.cumcount()
    df['customer_avg_price_before'] = g_cust['price_rur'].apply(lambda s: s.shift().expanding().mean()).droplevel(0)
    df['customer_n_unique_suppliers_before'] = g_cust['inn'].apply(n_unique_before).droplevel(0)
    df['customer_avg_price_drop_before'] = g_cust['price_drop_pct'].apply(lambda s: s.shift().expanding().mean()).droplevel(0)
    df['customer_supplier_concentration'] = np.where(
        df['customer_n_unique_suppliers_before'] > 0,
        1 / df['customer_n_unique_suppliers_before'],
        np.nan,
    )

    g_pair = df.groupby(['customer_inn', 'inn'], sort=False)
    df['pair_n_contracts_before'] = g_pair.cumcount()
    df['pair_total_sum_before'] = g_pair['price_rur'].apply(lambda s: s.shift().expanding().sum()).droplevel([0, 1])
    df['pair_avg_price_before'] = g_pair['price_rur'].apply(lambda s: s.shift().expanding().mean()).droplevel([0, 1])
    df['pair_days_since_last'] = g_pair['sign_date'].apply(lambda s: (s - s.shift()).dt.days).droplevel([0, 1])

    df['pair_share_in_supplier_portfolio'] = np.where(
        df['supplier_n_contracts_before'] > 0,
        df['pair_n_contracts_before'] / df['supplier_n_contracts_before'],
        np.nan,
    )
    df['pair_share_in_customer_portfolio'] = np.where(
        df['customer_n_contracts_before'] > 0,
        df['pair_n_contracts_before'] / df['customer_n_contracts_before'],
        np.nan,
    )

    df['price_vs_supplier_avg'] = np.where(
        df['supplier_avg_price_before'] > 0,
        df['price_rur'] / df['supplier_avg_price_before'],
        np.nan,
    )
    df['price_zscore_supplier'] = np.where(
        df['supplier_std_price_before'] > 0,
        (df['price_rur'] - df['supplier_avg_price_before']) / df['supplier_std_price_before'],
        np.nan,
    )
    df['price_drop_vs_supplier_avg'] = df['price_drop_pct'] - df['supplier_avg_price_drop_before']
    return df

dtype_map = {c: str for c in ID_COLUMNS}
df = pd.read_csv(INPUT_PATH, dtype=dtype_map, low_memory=False)
df = prepare(df)
df = add_contract_features(df)
df = add_history_features(df)
print(f"Total: {df.shape[0]} rows, {df.shape[1]} columns")
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}")

Start: 18537 rows, 96 columns
Contract-level features
History features
Total: 18537 rows, 116 columns
Saved features.csv


In [7]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

INPUT_PATH = '~/Downloads/files/features.csv'
FIG_DIR = Path('eda_figures')
FIG_DIR.mkdir(exist_ok=True, parents=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

C_NO = '#2ca02c'
C_YES = '#d62728'
PALETTE = {'No collusion': C_NO, 'Collusion': C_YES}

ID = ['regnum', 'notif_num', 'number', 'notification_purchase_number', 'purchase_code',
      'customer_spz_code', 'inn', 'customer_inn', 'customer_kpp', 'kpp', 'ogrn',
      'customer_okpo', 'suppliers_0_inn', 'suppliers_0_kpp']

def load():
    df = pd.read_csv(INPUT_PATH, dtype={c: str for c in ID}, low_memory=False)
    df['sign_date'] = pd.to_datetime(df['sign_date'])
    df['sign_year'] = df['sign_date'].dt.year
    df['label'] = df['collusion'].map({True: 'Collusion', False: 'No collusion'})
    return df

def fig_year(df):
    fig, ax = plt.subplots(figsize=(11, 5))
    yd = df.groupby(['sign_year', 'collusion']).size().unstack(fill_value=0)
    yd.columns = ['No collusion', 'Collusion']
    yd.plot(kind='bar', stacked=True, ax=ax, color=[C_NO, C_YES], width=0.8)
    ax.set_xlabel('Sign year')
    ax.set_ylabel('Number of contracts')
    ax.set_title('Contract distribution by year and collusion status')
    ax.legend(title='', loc='upper left')
    ax2 = ax.twinx()
    rate = df.groupby('sign_year')['collusion'].mean() * 100
    ax2.plot(range(len(rate)), rate.values, 'o-', color='black', linewidth=2, markersize=5, label='Collusion rate, %')
    ax2.set_ylabel('Collusion rate, %')
    ax2.legend(loc='upper center')
    ax2.grid(False)
    plt.xticks(rotation=45)
    plt.savefig(FIG_DIR / '01_year_distribution.png')
    plt.close()

def fig_fz_placing(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    fz = df.groupby(['fz', 'collusion']).size().unstack(fill_value=0)
    fz.columns = ['No collusion', 'Collusion']
    fz_pct = fz.div(fz.sum(axis=1), axis=0) * 100
    fz_pct.plot(kind='bar', stacked=True, ax=axes[0], color=[C_NO, C_YES])
    axes[0].set_title('Collusion share by federal law')
    axes[0].set_xlabel('Federal law')
    axes[0].set_ylabel('Share, %')
    axes[0].set_xticklabels([f'{i}-FZ' for i in fz.index], rotation=0)
    axes[0].legend(title='')
    for i, idx in enumerate(fz.index):
        tot = fz.loc[idx].sum()
        axes[0].text(i, 102, f'n={tot:,}', ha='center', fontsize=9)
    axes[0].set_ylim(0, 112)
    top = df['placing_way'].value_counts().head(8).index
    pw = df[df['placing_way'].isin(top)].groupby(['placing_way', 'collusion']).size().unstack(fill_value=0)
    pw.columns = ['No collusion', 'Collusion']
    pw_pct = pw.div(pw.sum(axis=1), axis=0) * 100
    pw_pct = pw_pct.sort_values('Collusion', ascending=False)
    pw_pct.plot(kind='bar', stacked=True, ax=axes[1], color=[C_NO, C_YES])
    axes[1].set_title('Collusion share by placement method (top 8)')
    axes[1].set_xlabel('Placement code')
    axes[1].set_ylabel('Share, %')
    axes[1].legend(title='', loc='lower right')
    plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '02_fz_placing.png')
    plt.close()

def fig_price_drop(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sub = df[df['price_drop_pct'].notna()].copy()
    sub = sub[(sub['price_drop_pct'] >= 0) & (sub['price_drop_pct'] <= 0.5)]
    for lbl, val, col in [('No collusion', False, C_NO), ('Collusion', True, C_YES)]:
        d = sub[sub['collusion'] == val]['price_drop_pct']
        axes[0].hist(d, bins=50, alpha=0.6, density=True, color=col, label=f'{lbl} (n={len(d):,})')
    axes[0].set_xlabel('Price drop from initial price, %')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Price drop distribution (0-50%)')
    axes[0].legend()
    for lbl, val, col in [('No collusion', False, C_NO), ('Collusion', True, C_YES)]:
        d = np.sort(sub[sub['collusion'] == val]['price_drop_pct'])
        axes[1].plot(d, np.arange(len(d)) / len(d), color=col, linewidth=2, label=f'{lbl} (median={np.median(d):.3f})')
    axes[1].set_xlabel('Price drop from initial price, %')
    axes[1].set_ylabel('Cumulative share')
    axes[1].set_title('Price drop, empirical CDF')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / '03_price_drop.png')
    plt.close()

def fig_price(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sub = df[df['price_rur'] > 0].copy()
    sub['log10_price'] = np.log10(sub['price_rur'])
    for lbl, val, col in [('No collusion', False, C_NO), ('Collusion', True, C_YES)]:
        d = sub[sub['collusion'] == val]['log10_price']
        axes[0].hist(d, bins=50, alpha=0.6, density=True, color=col, label=lbl)
    axes[0].set_xlabel('Contract value, log10(RUB)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Contract value distribution (log scale)')
    axes[0].legend()
    sub['price_q'] = pd.qcut(sub['price_rur'], 4, labels=['Q1\n(smallest)', 'Q2', 'Q3', 'Q4\n(largest)'])
    rate = sub.groupby('price_q')['collusion'].mean() * 100
    axes[1].bar(range(len(rate)), rate.values, color='#4c72b0')
    axes[1].set_xticks(range(len(rate)))
    axes[1].set_xticklabels(rate.index)
    axes[1].set_ylabel('Collusion rate, %')
    axes[1].set_title('Collusion rate by contract value quartile')
    for i, v in enumerate(rate.values):
        axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '04_contract_value.png')
    plt.close()

def fig_boxplots(df):
    feats = [
        ('customer_n_unique_suppliers_before', 'Unique suppliers of customer (before)'),
        ('customer_supplier_concentration', 'Customer supplier concentration'),
        ('pair_share_in_customer_portfolio', 'Pair share in customer portfolio'),
        ('supplier_avg_price_drop_before', 'Supplier avg price drop (before)'),
        ('customer_n_contracts_before', 'Customer contracts (before)'),
        ('n_suppliers', 'Number of suppliers on contract'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for i, (feat, title) in enumerate(feats):
        s = df[[feat, 'label']].dropna()
        lo, hi = s[feat].quantile([0.01, 0.99])
        s = s[(s[feat] >= lo) & (s[feat] <= hi)]
        sns.boxplot(data=s, x='label', y=feat, ax=axes[i], palette=PALETTE, showfliers=False, order=['No collusion', 'Collusion'])
        axes[i].set_title(title, fontsize=10)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
    plt.suptitle('Key behavioural features: collusion vs no collusion', y=1.0, fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '05_boxplots.png')
    plt.close()

def fig_heatmap_region_year(df):
    top_regions = df['contract_region_code'].value_counts().head(12).index
    sub = df[df['contract_region_code'].isin(top_regions)].copy()
    pivot = sub.pivot_table(index='contract_region_code', columns='sign_year', values='collusion', aggfunc='mean') * 100
    fig, ax = plt.subplots(figsize=(13, 6))
    sns.heatmap(pivot, cmap='Reds', annot=False, ax=ax, cbar_kws={'label': 'Collusion rate, %'}, linewidths=0.5, linecolor='white')
    ax.set_title('Collusion rate by region and year (top 12 regions by volume)')
    ax.set_xlabel('Sign year')
    ax.set_ylabel('Region code')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '06_heatmap_region_year.png')
    plt.close()

def fig_okpd(df):
    sub = df[df['okpd2_section'].notna()].copy()
    g = sub.groupby('okpd2_section').agg(n=('collusion', 'size'), rate=('collusion', 'mean')).reset_index()
    g = g[g['n'] >= 100].sort_values('rate', ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(g['okpd2_section'].astype(str)[::-1], (g['rate'] * 100)[::-1], color='#8856a7')
    ax.set_xlabel('Collusion rate, %')
    ax.set_ylabel('OKPD2 section')
    ax.set_title('Collusion rate by product category (OKPD2 section, n>=100)')
    for i, (_, row) in enumerate(g[::-1].iterrows()):
        ax.text(row['rate'] * 100 + 0.2, i, f"n={int(row['n'])}", va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '07_okpd_sections.png')
    plt.close()

def fig_concentration(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    cartel = df[df['collusion'] == True]
    sup = cartel['inn'].value_counts().head(15)
    axes[0].barh(range(len(sup))[::-1], sup.values, color=C_YES, alpha=0.8)
    axes[0].set_yticks(range(len(sup))[::-1])
    axes[0].set_yticklabels([s[:10] + '…' for s in sup.index], fontsize=8)
    axes[0].set_xlabel('Number of cartel contracts')
    axes[0].set_title('Top 15 suppliers by cartel contracts')
    cust = cartel['customer_inn'].value_counts().head(15)
    axes[1].barh(range(len(cust))[::-1], cust.values, color='#d95f0e', alpha=0.8)
    axes[1].set_yticks(range(len(cust))[::-1])
    axes[1].set_yticklabels([s[:10] + '…' for s in cust.index], fontsize=8)
    axes[1].set_xlabel('Number of cartel contracts')
    axes[1].set_title('Top 15 customers by cartel contracts')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '08_concentration.png')
    plt.close()

def fig_correlation(df):
    num_feats = ['price_drop_pct', 'log_price', 'n_suppliers', 'n_products',
                 'supplier_n_contracts_before', 'supplier_avg_price_before',
                 'supplier_avg_price_drop_before', 'customer_n_contracts_before',
                 'customer_n_unique_suppliers_before', 'customer_supplier_concentration',
                 'pair_n_contracts_before', 'pair_share_in_customer_portfolio',
                 'price_vs_supplier_avg', 'contract_duration_days']
    num_feats = [f for f in num_feats if f in df.columns]
    corr = df[num_feats].corr()
    fig, ax = plt.subplots(figsize=(11, 9))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7}, square=True, cbar_kws={'shrink': 0.7})
    ax.set_title('Correlation matrix of key numeric features')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '09_correlation.png')
    plt.close()

def fig_missing(df):
    model_feats = [c for c in df.columns if c.startswith(('supplier_', 'customer_', 'pair_',
                   'price_', 'days_', 'n_', 'is_', 'sign_')) or c in
                   ['notification_max_price', 'okpd2_section', 'placing_way', 'opf_code',
                    'industry', 'subindustry', 'contract_duration_days', 'log_price']]
    miss = df[model_feats].isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss) * 0.32)))
    ax.barh(range(len(miss))[::-1], (miss.values * 100)[::-1],
            color=['#d62728' if v > 0.5 else '#1f77b4' for v in miss.values[::-1]])
    ax.set_yticks(range(len(miss))[::-1])
    ax.set_yticklabels(miss.index[::-1], fontsize=8)
    ax.set_xlabel('Missing values, %')
    ax.set_title('Missingness by feature (only features with missing values), red = >50%')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '10_missingness.png')
    plt.close()

def single_feature_auc(df):
    y = df['collusion'].astype(int).values
    candidates = [
        'customer_supplier_concentration', 'customer_n_unique_suppliers_before',
        'pair_share_in_customer_portfolio', 'pair_n_contracts_before',
        'supplier_avg_price_drop_before', 'price_drop_pct', 'price_drop_vs_supplier_avg',
        'price_vs_supplier_avg', 'price_zscore_supplier', 'log_price',
        'supplier_n_contracts_before', 'customer_n_contracts_before',
        'supplier_n_unique_customers_before', 'n_products', 'contract_duration_days',
    ]
    candidates = [f for f in candidates if f in df.columns]
    rows = []
    for f in candidates:
        x = pd.to_numeric(df[f], errors='coerce')
        m = x.notna().values
        auc = roc_auc_score(y[m], x[m].values)
        rows.append((f, max(auc, 1 - auc)))
    rows.sort(key=lambda r: r[1], reverse=True)
    print("Single-feature AUC (discriminative power, collusion vs feature)")
    for f, auc in rows:
        print(f"{f}: AUC {auc:.3f}")
    return rows

def overview_stats(df):
    return {
        'total': len(df),
        'collusion': int(df['collusion'].sum()),
        'rate': float(df['collusion'].mean()),
        'suppliers': int(df['inn'].nunique()),
        'customers': int(df['customer_inn'].nunique()),
        'date_min': str(df['sign_date'].min().date()),
        'date_max': str(df['sign_date'].max().date()),
        'fz44': int((df['fz'].astype(str) == '44').sum()),
        'fz223': int((df['fz'].astype(str) == '223').sum()),
    }

df = load()
stats = overview_stats(df)
for k, v in stats.items():
    print(f"{k}: {v}")

single_feature_auc(df)

print("Generating figures")
fig_year(df)
fig_fz_placing(df)
fig_price_drop(df)
fig_price(df)
fig_boxplots(df)
fig_heatmap_region_year(df)
fig_okpd(df)
fig_concentration(df)
fig_correlation(df)
fig_missing(df)
print(f"Figures saved to {FIG_DIR}")

total: 18537
collusion: 1471
rate: 0.07935480390570211
suppliers: 184
customers: 4518
date_min: 2010-12-24
date_max: 2025-12-30
fz44: 16989
fz223: 1548
Single-feature AUC (discriminative power, collusion vs feature)
customer_supplier_concentration: AUC 0.729
customer_n_unique_suppliers_before: AUC 0.685
pair_share_in_customer_portfolio: AUC 0.666
customer_n_contracts_before: AUC 0.660
price_vs_supplier_avg: AUC 0.652
supplier_avg_price_drop_before: AUC 0.616
log_price: AUC 0.601
supplier_n_unique_customers_before: AUC 0.592
price_zscore_supplier: AUC 0.587
pair_n_contracts_before: AUC 0.580
price_drop_pct: AUC 0.564
n_products: AUC 0.562
supplier_n_contracts_before: AUC 0.550
contract_duration_days: AUC 0.549
price_drop_vs_supplier_avg: AUC 0.529
Generating figures
Figures saved to eda_figures
